In [1]:
!pip install pandas numpy tqdm pyarrow

In [1]:
import pandas as pd
import numpy as np
import glob
import os
from tqdm import tqdm

In [3]:
archivos_input=(
    glob.glob("C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/CSV-01-12/01-12/*.csv") + 
    glob.glob("C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/CSV-03-11/03-11/*.csv")
)

Schema={
    " Total Fwd Packets" : "int32",
    " Fwd Header Length" : "int64",
    " Bwd Header Length" : "int64",
    "Subflow Fwd Packets" : "int32",
    " act_data_pkt_fwd" : "int16",
    " min_seg_size_forward" : "int32"
}

def shrink_df(df: pd.DataFrame,skip=None)->pd.DataFrame:

    if skip is None:
        skip=set()
    elif isinstance(skip,str):
        skip={skip}
    else:
        skip=set(skip)
    
    for col in df.columns:
        if col in skip:
            continue

        s=df[col]
        col_type=s.dtype

        if pd.api.types.is_datetime64_any_dtype(col_type) or pd.api.types.is_timedelta64_dtype(col_type) or isinstance(col_type, pd.CategoricalDtype):
            continue

        if pd.api.types.is_bool_dtype(col_type):
            continue

        elif pd.api.types.is_integer_dtype(col_type):
            c_min=s.min()
            c_max=s.max()

            if np.iinfo(np.int8).min<=c_min<=c_max<=np.iinfo(np.int8).max:
                df[col]=s.astype(np.int8)
            elif np.iinfo(np.int16).min<=c_min<=c_max<=np.iinfo(np.int16).max:
                df[col]=s.astype(np.int16)
            elif np.iinfo(np.int32).min<=c_min<=c_max<=np.iinfo(np.int32).max:
                df[col]=s.astype(np.int32)
            else:
                df[col]=s.astype(np.int64)

        elif pd.api.types.is_float_dtype(col_type):
            c_min=s.min()
            c_max=s.max()

            if np.finfo(np.float32).min<=c_min<=c_max<=np.finfo(np.float32).max:
                df[col]=s.astype(np.float32)
            else:
                df[col]=s.astype(np.float64)
    
    return df

def apply_schema(df,schema):
    for col,dtype in schema.items():
        if col in df.columns:
            df[col]=df[col].astype(dtype)
    return df

columns_drop=[
    "Unnamed: 0",
    "Flow ID",
    " Source IP",
    " Source Port",
    " Destination IP",
    " Destination Port",
    " Timestamp",
    " Bwd PSH Flags",
    " Fwd URG Flags",
    " Bwd URG Flags",
    "FIN Flag Count",
    " PSH Flag Count",
    " ECE Flag Count",
    "Fwd Avg Bytes/Bulk",
    " Fwd Avg Packets/Bulk",
    " Fwd Avg Bulk Rate",
    " Bwd Avg Bytes/Bulk",
    " Bwd Avg Packets/Bulk",
    "Bwd Avg Bulk Rate",
    " Fwd Header Length.1",
    " Inbound",
    "SimillarHTTP"
]

for file in tqdm(archivos_input):
    nombre=os.path.basename(file)
    print(f"Leyendo: {nombre}")

    df=pd.read_csv(file,usecols=lambda column: column not in columns_drop)
    #inicial_mem=df.memory_usage().sum()
    #print("Uso inicial de memoria: ",inicial_mem,"MB")
    df=df.replace([np.inf,-np.inf],np.nan)
    df.dropna(inplace=True)
    df=shrink_df(df,skip=[' Label'])
    df=apply_schema(df,Schema)
    df.drop_duplicates(inplace=True)
    #final_mem=df.memory_usage().sum()
    #print("Uso final de memoria: ",final_mem,"MB")

    basename=nombre.split('.')[0]
    df.to_parquet(f"C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/Parquet/{basename}.parquet")
    print(file,':\n',df[' Label'].value_counts())

  0%|                                                                                            | 0/9 [00:00<?, ?it/s]

Leyendo: DrDoS_LDAP.csv


 11%|█████████▎                                                                          | 1/9 [00:42<05:40, 42.58s/it]

C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/CSV-01-12/01-12\DrDoS_LDAP.csv :
  Label
DrDoS_LDAP    28843
BENIGN         1391
Name: count, dtype: int64
Leyendo: DrDoS_MSSQL.csv


 22%|██████████████████▋                                                                 | 2/9 [02:15<08:24, 72.07s/it]

...

C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/CSV-03-11/03-11\Syn.csv :
  Label
Syn       452865
BENIGN     27034
Name: count, dtype: int64
Leyendo: UDP.csv


 89%|██████████████████████████████████████████████████████████████████████████▋         | 8/9 [10:14<01:30, 90.32s/it]

C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/CSV-03-11/03-11\UDP.csv :
  Label
UDP       1288161
MSSQL        4951
BENIGN       2833
Name: count, dtype: int64
Leyendo: UDPLag.csv


100%|████████████████████████████████████████████████████████████████████████████████████| 9/9 [10:30<00:00, 70.07s/it]

C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/CSV-03-11/03-11\UDPLag.csv :
  Label
Syn       79050
UDP       56390
BENIGN     3748
UDPLag      378
Name: count, dtype: int64


In [4]:
file="C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/Parquet/MSSQL.parquet"
df=pd.read_parquet(file)

In [5]:
res=df.columns.to_list()
print("Lista de columnas:",'\n')
print(res)
print('\n',"Número de columnas:")
print(len(res))

Lista de columnas: 

[' Protocol', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Variance', ' SYN Flag Count', ' RST Flag Count', ' ACK Flag Count', ' URG Flag Count', ' CWE Flag Count', ' Down/Up Ratio', ' Average Packet Size', ' Avg Fwd

In [6]:
for i in range(101):
    row = [i]
    print(df.loc[df.index[row]],'\n')

    Protocol   Flow Duration   Total Fwd Packets   Total Backward Packets  \
0         17               3                   2                        0   

   Total Length of Fwd Packets   Total Length of Bwd Packets  \
0                       2944.0                           0.0   

    Fwd Packet Length Max   Fwd Packet Length Min   Fwd Packet Length Mean  \
0                  1472.0                  1472.0                   1472.0   

    Fwd Packet Length Std  ...   min_seg_size_forward  Active Mean  \
0                     0.0  ...                     -1          0.0   

    Active Std   Active Max   Active Min  Idle Mean   Idle Std   Idle Max  \
0          0.0          0.0          0.0        0.0        0.0        0.0   

    Idle Min   Label  
0        0.0    LDAP  

[1 rows x 66 columns] 

    Protocol   Flow Duration   Total Fwd Packets   Total Backward Packets  \
1          0       117876168               25274                       50   

   Total Length of Fwd Packets   Tota

In [7]:
df=pd.read_parquet("C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/Parquet/DrDoS_UDP.parquet")

df[' Label']=df[' Label'].replace("DrDoS_UDP","UDP")

df.to_parquet("C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/Parquet/DrDoS_UDP.parquet")

In [8]:
df=pd.read_parquet("C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/Parquet/DrDoS_LDAP.parquet")

df[' Label']=df[' Label'].replace("DrDoS_LDAP","LDAP")

df.to_parquet("C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/Parquet/DrDoS_LDAP.parquet")

In [9]:
df=pd.read_parquet("C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/Parquet/DrDoS_MSSQL.parquet")

df[' Label']=df[' Label'].replace("DrDoS_MSSQL","MSSQL")

df.to_parquet("C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/Parquet/DrDoS_MSSQL.parquet")

In [10]:
!pip install hvplot pandas pyarrow

In [11]:
import pandas as pd
import hvplot.pandas

import holoviews as hv
hv.extension('bokeh')

%opts magic unavailable (pyparsing cannot be imported)
%compositor magic unavailable (pyparsing cannot be imported)


In [12]:
import glob

files=glob.glob("C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/Parquet/*.parquet")

df=pd.concat([pd.read_parquet(f,columns=[' Label']) for f in files],ignore_index=True)

In [13]:
counts=df[' Label'].value_counts().reset_index()
counts.columns=['Label','Count']

In [14]:
counts.hvplot.bar(
    x='Label',
    y='Count',
    title="Distribución de Labels",
    rot=45
)

:Bars   [Label]   (Count)

In [15]:
df=pd.read_parquet("C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/Parquet/LDAP.parquet")

df=df.drop(df[df[" Label"].isin(["NetBIOS","UDPLag"])].index)

df.to_parquet("C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/Parquet/LDAP.parquet")

In [16]:
df=pd.read_parquet("C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/Parquet/UDPLag.parquet")

df=df.drop(df[df[" Label"].isin(["NetBIOS","UDPLag"])].index)

df.to_parquet("C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/Parquet/UDPLag.parquet")

In [17]:
!pip install scikit-learn

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

In [3]:
#########################################
#Dataframe
#########################################
rutas=glob.glob("C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/Parquet/*.parquet")
dfs=[pd.read_parquet(r) for r in rutas]
df=pd.concat(dfs,ignore_index=True)

In [4]:
#########################################
#Resample
#########################################
UDP_df=df[df[" Label"]=="UDP"]
LDAP_df=df[df[" Label"]=="LDAP"]
MSSQL_df=df[df[" Label"]=="MSSQL"]
Syn_df=df[df[" Label"]=="Syn"]
Benign_df=df[df[" Label"]=="BENIGN"]

n_objetivo=20000

UDP_bal=resample(UDP_df,replace=False,n_samples=n_objetivo,random_state=42)
LDAP_bal=resample(LDAP_df,replace=False,n_samples=n_objetivo,random_state=42)
MSSQL_bal=resample(MSSQL_df,replace=False,n_samples=n_objetivo,random_state=42)
Syn_bal=resample(Syn_df,replace=False,n_samples=n_objetivo,random_state=42)
df=pd.concat([UDP_bal,LDAP_bal,MSSQL_bal,Syn_bal,Benign_df],ignore_index=True)

df.to_parquet(f"C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/Training/full.parquet")

In [6]:
import glob

file="C:/Users/sergi/Desktop/TFGSergioFernandezCamara/Files/Training/full.parquet"

dfPlot=pd.read_parquet(file,columns=[' Label'])

In [49]:
counts=dfPlot[' Label'].value_counts().reset_index()
counts.columns=['Label','Count']

In [50]:
counts.hvplot.bar(
    x='Label',
    y='Count',
    title="Distribución de Labels",
    rot=45
)

:Bars   [Label]   (Count)

In [5]:
#########################################
#Label Encode
#########################################
df["LabelBin"]=(df[" Label"]!="BENIGN").astype("int32")
df=df.drop(columns=[" Label"])

In [6]:
#########################################
#Divide Train, Val y Test
#########################################
x=df.drop(columns=["LabelBin"])
y=df["LabelBin"]

x_train,x_temp,y_train,y_temp=train_test_split(
    x,y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

x_val,x_test,y_val,y_test=train_test_split(
    x_temp,y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:",x_train.shape,y_train.shape)
print("Validation:",x_val.shape,y_val.shape)
print("Test:",x_test.shape,y_test.shape)

Train: (88516, 65) (88516,)
Validation: (18968, 65) (18968,)
Test: (18968, 65) (18968,)


In [53]:
!pip install tensorflow

In [7]:
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
import keras
from keras import layers

In [8]:
train_df=x_train.copy()
train_df["LabelBin"]=y_train.copy()

train_df=train_df.sample(frac=1,random_state=42).reset_index(drop=True)

x_train=train_df.drop(columns=["LabelBin"]).astype("float32").to_numpy()
y_train=train_df["LabelBin"].astype("float32").to_numpy().reshape(-1, 1)

x_val = x_val.astype("float32").to_numpy()
y_val = y_val.astype("float32").to_numpy().reshape(-1, 1)

x_test = x_test.astype("float32").to_numpy()
y_test = y_test.astype("float32").to_numpy().reshape(-1, 1)

In [9]:
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_val=scaler.transform(x_val)
x_test=scaler.transform(x_test)

In [10]:
model=keras.Sequential([
    keras.Input(shape=(x_train.shape[1],)),
    layers.Dense(64,activation="relu"),
    layers.Dense(64,activation="relu"),
    layers.Dense(1,activation="sigmoid"),
])

In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 64)                  │           4,224 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 64)                  │           4,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 8,449 (33.00 KB)

 Trainable params: 8,449 (33.00 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[keras.metrics.BinaryCrossentropy(),
             keras.metrics.BinaryAccuracy(),
             keras.metrics.Precision(thresholds=0.5),
             keras.metrics.Recall(thresholds=0.5),
             keras.metrics.AUC(),
             keras.metrics.F1Score(average="micro",threshold=0.5)],
)

In [21]:
my_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        min_delta=1e-2,
        patience=2,
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        filepath="mejor_modelo.keras",
        monitor="val_loss",
        mode="min",
        save_best_only=True
    ),
]

In [22]:
history=model.fit(
    x_train,
    y_train,
    batch_size=32,
    epochs=20,
    validation_data=(x_val,y_val),
    callbacks=my_callbacks
)

Epoch 1/20
2767/2767 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - auc_2: 0.9997 - binary_accuracy: 0.9978 - binary_crossentropy: 0.0089 - f1_score: 0.9983 - loss: 0.0089 - precision_2: 0.9993 - recall_2: 0.9973 - val_auc_2: 0.9995 - val_binary_accuracy: 0.9977 - val_binary_crossentropy: 0.0149 - val_f1_score: 0.9982 - val_loss: 0.0149 - val_precision_2: 0.9993 - val_recall_2: 0.9970
Epoch 2/20
2767/2767 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - auc_2: 0.9997 - binary_accuracy: 0.9980 - binary_crossentropy: 0.0082 - f1_score: 0.9984 - loss: 0.0082 - precision_2: 0.9994 - recall_2: 0.9973 - val_auc_2: 0.9996 - val_binary_accuracy: 0.9971 - val_binary_crossentropy: 0.0099 - val_f1_score: 0.9977 - val_loss: 0.0099 - val_precision_2: 0.9990 - val_recall_2: 0.9964
Epoch 3/20
2767/2767 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - auc_2: 0.9997 - binary_accuracy: 0.9981 - binary_crossentropy: 0.0082 - f1_score: 0.9985 - loss: 0.0082 - precision_2: 0.9995 - recall_2: 0.9974 - val_auc_2: 0.9996 - val_binary_accuracy: 0.9

In [23]:
model.weights

[<Variable path=sequential/dense/kernel, shape=(65, 64), dtype=float32, value=[[ 0.04947152  0.0174302   0.04499593 ...  0.37534338  0.20389302
   -0.45644015]
  [-0.07364921 -0.27134752 -0.0950199  ...  0.00192618  0.14277267
    0.32257923]
  [ 0.01642776  0.09057025  0.27273807 ...  0.07507371  0.14524058
   -0.05662685]
  ...
  [ 0.04854115 -0.11251434  0.25136897 ...  0.12152538 -0.18249933
    0.14836308]
  [ 0.1158344  -0.1480005   0.23506403 ... -0.11132647 -0.18521991
    0.14567807]
  [-0.11247011 -0.00516794 -0.28664118 ...  0.17540762 -0.22078548
    0.20271225]]>,
 <Variable path=sequential/dense/bias, shape=(64,), dtype=float32, value=[-0.02779886  0.01427241  0.02147068  0.04558962 -0.10846964 -0.02240675
  -0.02458062 -0.2957563   0.14941281 -0.23191343 -0.0443045  -0.11525511
   0.06254169 -0.06937148 -0.0770696  -0.02690839 -0.218779    0.13229021
   0.00269377 -0.02192806 -0.0943846   0.0240203  -0.10069773  0.08740066
  -0.2097618   0.03236129 -0.04289633 -0.0575343

In [24]:
print(history.history)

{'auc_2': [0.999720573425293, 0.999743640422821, 0.999697744846344], 'binary_accuracy': [0.9978309273719788, 0.9979551434516907, 0.9980568289756775], 'binary_crossentropy': [0.008928266353905201, 0.00823774840682745, 0.008160202763974667], 'f1_score': [0.9982839822769165, 0.9983822107315063, 0.9984626770019531], 'loss': [0.008928266353905201, 0.00823774840682745, 0.008160202763974667], 'precision_2': [0.9992664456367493, 0.9994273781776428, 0.9994989633560181], 'recall_2': [0.9973035454750061, 0.9973393082618713, 0.9974285960197449], 'val_auc_2': [0.9995273351669312, 0.9995933771133423, 0.9996084570884705], 'val_binary_accuracy': [0.9976803064346313, 0.9971003532409668, 0.9976803064346313], 'val_binary_crossentropy': [0.014870780520141125, 0.009901454672217369, 0.010543872602283955], 'val_f1_score': [0.9981645345687866, 0.997705340385437, 0.9981642365455627], 'val_loss': [0.014870780520141125, 0.009901454672217369, 0.010543872602283955], 'val_precision_2': [0.9993317723274231, 0.998997

In [25]:
results=model.evaluate(x_test,y_test)
print("loss, Bcross, Bacc, Prec, Recall, AUC, F1S: ",results)

593/593 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - auc_2: 0.9995 - binary_accuracy: 0.9975 - binary_crossentropy: 0.0109 - f1_score: 0.9980 - loss: 0.0109 - precision_2: 0.9997 - recall_2: 0.9963
loss, Bcross, Bacc, Prec, Recall, AUC, F1S:  [0.010907653719186783, 0.010907653719186783, 0.997469425201416, 0.9996655583381653, 0.9963333606719971, 0.9994965195655823, 0.9979966878890991]


In [26]:
model.save("model.keras")